In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoConfig, BitsAndBytesConfig, AutoModelForCausalLM, TextStreamer, TrainingArguments, Trainer
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from datasets import Dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModelForCausalLM
)
from trl import SFTTrainer

c:\Users\omen\Desktop\projects\gemma4-darija-dz\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# login to Hugging Face using the token from the .env file
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Warning: HF_TOKEN not found in environment")
model_id = "google/gemma-4-E2B"
device_map = {"": 0}
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

In [ ]:
# Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["vision_tower", "audio_tower"],
)

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_id, extra_special_tokens={})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

streamer = TextStreamer(tokenizer, skip_prompt=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={'': 0},
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    attn_implementation="sdpa",  # Accelerated PyTorch Scaled Dot-Product Attention
)


In [ ]:
# skipped (diagnostic only)

In [ ]:
# skipped (diagnostic only)

In [ ]:
# skipped (diagnostic only)

In [ ]:
print("Model loaded successfully. Skipping full model print to save memory.")

In [ ]:
# Bypass prepare_model_for_kbit_training to avoid OOM on Gemma4's
# embed_tokens_per_layer (262144 x 8960 bf16 = ~4.7GB → fp32 = ~9.4GB → OOM)

# Step 1: Disable KV cache (required for gradient checkpointing)
model.config.use_cache = False

# Step 2: Enable gradient checkpointing directly
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Step 3: Selectively upcast small non-4bit params to float32
# (skip large embeddings that would exceed 8GB VRAM)
SKIP_UPCAST = {"embed_tokens_per_layer", "embed_tokens", "lm_head"}
for name, param in model.named_parameters():
    if param.__class__.__name__ == "Params4bit":
        continue  # quantized weights — leave as-is
    if param.dtype not in (torch.float16, torch.bfloat16):
        continue  # already float32 or other dtype
    if any(skip in name for skip in SKIP_UPCAST):
        continue  # too large to safely upcast on 8GB VRAM
    param.data = param.data.to(torch.float32)

# Step 4: Freeze all base-model parameters
for param in model.parameters():
    param.requires_grad_(False)

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after prep: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=r"model\.language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
df = pd.read_csv("arabic_v1.csv")
if "Text" in df.columns and "text" not in df.columns:
    df = df.rename(columns={"Text": "text"})
dataset = Dataset.from_pandas(df[["text"]].dropna())
splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./gemma4-darija-qlora",
    dataset_text_field="text",
    max_length=512,
    packing=True,                   # Dense 512-token packing (cuts 137k rows down to ~7.3k dense sequences)
    dataset_num_proc=4,             # Parallel preprocessing
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16 packed chunks (~350-400 sentences/step)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=1,             # Full real training for 1 complete epoch over the entire dataset
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,                  # Periodic evaluation on validation split
    save_steps=50,                  # Save LoRA checkpoints regularly
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none"
)


In [ ]:
# model is already a PeftModel from get_peft_model() in cell 10.
# TRL 1.10 raises if you pass both a PeftModel AND peft_config — so omit peft_config.
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)


In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("gemma4-darija-qlora")
tokenizer.save_pretrained("gemma4-darija-qlora")
print("Training finished & LoRA adapters saved!")

In [ ]:
# Load base model + LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map=device_map,
    torch_dtype=compute_dtype
)
model_with_adapter = PeftModelForCausalLM.from_pretrained(base_model, "./gemma4-darija-qlora")
prompt = "واش راك اليوم؟" # "How are you today?" in Darija
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model_with_adapter.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
